In [ ]:
import pyearthtools.pipeline
import pyearthtools.data
import pyearthtools.tutorial

import os
os.environ['PETPROJECT'] = os.path.expanduser("~") + '/dev/proj/petcache'
workdir = os.environ['PETPROJECT']
file_location = workdir + '/mini.nc'

p1 = pyearthtools.pipeline.Pipeline(
    pyearthtools.tutorial.ERA5DataClass.ERA5LowResDemoIndex([
        '10m_u_component_of_wind', 
        '10m_v_component_of_wind', 
        'mean_sea_level_pressure',
        '2m_temperature'    
        ], filename_override=file_location),
    pyearthtools.pipeline.operations.xarray.Sort(
        ["2m_temperature", "u_component_of_wind", "v_component_of_wind", "vorticity", "geopotential"]
    ),
    pyearthtools.data.transforms.coordinates.StandardLongitude(type="0-360"),
    name="make_flat_xarray"
)

In [ ]:
p2 = pyearthtools.pipeline.Pipeline(
    pyearthtools.pipeline.modifications.TemporalRetrieval(
        concat=True, samples=((0, 1), (6, 1, 6)) # Input = 1 sample from time T=0 hours. Output = T+6,+12,+18,+24
    ),     
    pyearthtools.pipeline.operations.xarray.normalisation.MagicNorm(cache_dir=workdir),  # Incremental normalisation calculator    
    pyearthtools.pipeline.operations.xarray.conversion.ToNumpy(),
    pyearthtools.pipeline.operations.numpy.reshape.Rearrange('c t h w -> t h w c'), # channel batch height width -> batch height width channel
    name='to_netcdf'
)

In [ ]:
p3 = p1 | p2

In [ ]:
p3.named['make_flat_xarray']['20220202T00']

In [ ]:
p3['20220202T00']